# Ablation Study: Adding Freight to the Model

**Question:** Does adding `freight_mean` improve demand prediction?

**Experiment:**
1. Baseline: current model (without freight)
2. Test: model with freight_mean added to context features

**Metric:** Test corr_mse

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Load data
DATA_DIR = Path('../data/processed')
panel = pd.read_csv(DATA_DIR / 'panel.csv')
print(f"Panel shape: {panel.shape}")
print(f"\nfreight_mean stats:")
print(panel['freight_mean'].describe())

In [ ]:
# Feature definitions - WITHOUT freight (baseline)
CONTEXT_FEATURES_BASE = [
    'year', 'month', 'weekofyear', 'week_sin', 'week_cos',
    'demand_lag_1', 'demand_lag_2', 'demand_roll_4',
    'price_lag_1', 'price_roll_4',
    'weeks_since_last_sale',
    'price_std', 'price_range'
]

# Feature definitions - WITH freight
CONTEXT_FEATURES_FREIGHT = CONTEXT_FEATURES_BASE + ['freight_mean']

PRODUCT_FEATURES = [
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm',
    'product_photos_qty', 'product_name_length', 'product_description_length'
]

REVIEW_FEATURES = ['sku_review_count', 'sku_review_mean', 'sku_share_low']

PRICE_FEATURE = 'r_clipped'
TARGET = 'y'

print(f"Context features (base): {len(CONTEXT_FEATURES_BASE)}")
print(f"Context features (with freight): {len(CONTEXT_FEATURES_FREIGHT)}")

In [ ]:
# Preprocessing
for col in ['demand_lag_1', 'demand_lag_2', 'demand_roll_4', 'price_lag_1', 'price_roll_4', 'weeks_since_last_sale']:
    panel[col] = panel[col].fillna(0)

for col in REVIEW_FEATURES:
    panel[col] = panel[col].fillna(0)

for col in PRODUCT_FEATURES:
    panel[col] = panel[col].fillna(panel[col].median())

# Fill freight_mean NaN with median
panel['freight_mean'] = panel['freight_mean'].fillna(panel['freight_mean'].median())

# Encode categories
le_category = LabelEncoder()
panel['category_code'] = le_category.fit_transform(panel['product_category_name_english'].fillna('unknown'))
n_categories = len(le_category.classes_)

le_product = LabelEncoder()
panel['product_code'] = le_product.fit_transform(panel['product_id'])
n_products = len(le_product.classes_)

print(f"Categories: {n_categories}, Products: {n_products}")

In [ ]:
# Split data
train_df = panel[panel['split'] == 'train'].copy()
val_df = panel[panel['split'] == 'val'].copy()
test_df = panel[panel['split'] == 'test'].copy()

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
# Model components (same as nn_improved)

class ContextEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
    
    def forward(self, x):
        return self.net(x)


class MonotonicPriceEncoder(nn.Module):
    def __init__(self, z_dim, num_basis=20, hidden_dim=128):
        super().__init__()
        self.num_basis = num_basis
        self.z_encoder = nn.Sequential(
            nn.Linear(z_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 3 * num_basis)
        )
        
    def forward(self, log_r, z, return_at_ref=False):
        B = log_r.shape[0]
        params = self.z_encoder(z).view(B, self.num_basis, 3)
        
        a = F.softplus(params[:, :, 0]) + 0.1
        b = params[:, :, 1]
        w = -F.softplus(params[:, :, 2])
        
        log_r_expanded = log_r.unsqueeze(1)
        g = (w * torch.sigmoid(a * log_r_expanded + b)).sum(dim=1, keepdim=True)
        
        if return_at_ref:
            g_ref = (w * torch.sigmoid(b)).sum(dim=1, keepdim=True)
            return g, g_ref
        return g


class TwoHeadModel(nn.Module):
    def __init__(self, context_dim, product_dim, review_dim, n_categories, n_products,
                 category_embedding_dim=16, product_embedding_dim=16,
                 context_hidden=256, price_hidden=128, num_basis=20, dropout=0.15):
        super().__init__()
        
        self.category_embedding = nn.Embedding(n_categories, category_embedding_dim)
        self.product_embedding = nn.Embedding(n_products, product_embedding_dim)
        
        context_input_dim = context_dim + category_embedding_dim + product_embedding_dim
        self.context_encoder = ContextEncoder(context_input_dim, context_hidden, dropout)
        
        z_dim = category_embedding_dim + product_embedding_dim + product_dim + review_dim
        self.price_encoder = MonotonicPriceEncoder(z_dim, num_basis, price_hidden)
    
    def forward(self, batch):
        cat_emb = self.category_embedding(batch['category'])
        prod_emb = self.product_embedding(batch['product_id'])
        
        context_input = torch.cat([batch['context'], cat_emb, prod_emb], dim=1)
        log_demand_base = self.context_encoder(context_input)
        
        z = torch.cat([cat_emb, prod_emb, batch['product'], batch['review']], dim=1)
        g_r, g_ref = self.price_encoder(batch['log_r'], z, return_at_ref=True)
        log_mul = g_r - g_ref
        
        return (log_demand_base + log_mul).squeeze(1)

In [ ]:
# Dataset class
class DemandDataset(Dataset):
    def __init__(self, df, context_arr, product_arr, review_arr):
        self.context = torch.FloatTensor(context_arr)
        self.product = torch.FloatTensor(product_arr)
        self.review = torch.FloatTensor(review_arr)
        self.r = torch.FloatTensor(df[PRICE_FEATURE].values)
        self.log_r = torch.log(self.r)
        self.category = torch.LongTensor(df['category_code'].values)
        self.product_id = torch.LongTensor(df['product_code'].values)
        self.y = torch.FloatTensor(df[TARGET].values)
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return {
            'context': self.context[idx],
            'product': self.product[idx],
            'review': self.review[idx],
            'r': self.r[idx],
            'log_r': self.log_r[idx],
            'category': self.category[idx],
            'product_id': self.product_id[idx],
            'y': self.y[idx]
        }

In [ ]:
# Evaluation functions
def compute_corr_mse(y_true, y_pred, groups):
    df = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred, 'group': groups})
    
    def group_corr_mse(g):
        if len(g) < 2:
            return 0, 0
        y_t = g['y_true'].values
        y_p = g['y_pred'].values
        y_t_centered = y_t - y_t.mean()
        y_p_centered = y_p - y_p.mean()
        mse = ((y_t_centered - y_p_centered) ** 2).mean()
        return mse * len(g), len(g)
    
    results = df.groupby('group').apply(group_corr_mse)
    total_mse = sum(r[0] for r in results)
    total_weight = sum(r[1] for r in results)
    
    return total_mse / total_weight if total_weight > 0 else 0


def evaluate_model(model, loader, df, device):
    model.eval()
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            preds = model(batch)
            all_preds.append(preds.cpu().numpy())
            all_targets.append(batch['y'].cpu().numpy())
    
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)
    
    return {
        'corr_mse': compute_corr_mse(y_true, y_pred, df['product_id'].values),
        'r2': r2_score(y_true, y_pred)
    }, y_pred

In [ ]:
# Training function
def train_model(model, train_loader, val_loader, train_df, val_df, device,
                epochs=200, lr=5e-4, weight_decay=1e-5, patience=40, grad_clip=1.0):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.MSELoss()
    
    best_val_corr_mse = float('inf')
    best_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            preds = model(batch)
            loss = criterion(preds, batch['y'])
            loss.backward()
            if grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
        
        scheduler.step()
        
        val_metrics, _ = evaluate_model(model, val_loader, val_df, device)
        
        if val_metrics['corr_mse'] < best_val_corr_mse:
            best_val_corr_mse = val_metrics['corr_mse']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}: val_corr_mse={val_metrics['corr_mse']:.4f} (best={best_val_corr_mse:.4f})")
        
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    if best_state:
        model.load_state_dict(best_state)
    
    return model, best_val_corr_mse

In [ ]:
def run_experiment(context_features, experiment_name):
    """Run a single experiment with given context features."""
    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {experiment_name}")
    print(f"Context features: {len(context_features)}")
    print(f"{'='*60}")
    
    # Standardize features
    scaler_context = StandardScaler()
    scaler_product = StandardScaler()
    scaler_review = StandardScaler()
    
    train_context = scaler_context.fit_transform(train_df[context_features])
    train_product = scaler_product.fit_transform(train_df[PRODUCT_FEATURES])
    train_review = scaler_review.fit_transform(train_df[REVIEW_FEATURES])
    
    val_context = scaler_context.transform(val_df[context_features])
    val_product = scaler_product.transform(val_df[PRODUCT_FEATURES])
    val_review = scaler_review.transform(val_df[REVIEW_FEATURES])
    
    test_context = scaler_context.transform(test_df[context_features])
    test_product = scaler_product.transform(test_df[PRODUCT_FEATURES])
    test_review = scaler_review.transform(test_df[REVIEW_FEATURES])
    
    # Handle NaN
    for arr in [train_context, train_product, train_review, 
                val_context, val_product, val_review,
                test_context, test_product, test_review]:
        arr[np.isnan(arr)] = 0
    
    # Create datasets
    train_dataset = DemandDataset(train_df, train_context, train_product, train_review)
    val_dataset = DemandDataset(val_df, val_context, val_product, val_review)
    test_dataset = DemandDataset(test_df, test_context, test_product, test_review)
    
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)
    
    # Create model
    model = TwoHeadModel(
        context_dim=len(context_features),
        product_dim=len(PRODUCT_FEATURES),
        review_dim=len(REVIEW_FEATURES),
        n_categories=n_categories,
        n_products=n_products
    )
    
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {n_params:,}")
    
    # Train
    model, best_val = train_model(model, train_loader, val_loader, train_df, val_df, device)
    
    # Evaluate on test
    test_metrics, _ = evaluate_model(model, test_loader, test_df, device)
    
    print(f"\nResults:")
    print(f"  Test corr_mse: {test_metrics['corr_mse']:.4f}")
    print(f"  Test R²: {test_metrics['r2']:.4f}")
    
    return {
        'experiment': experiment_name,
        'n_context_features': len(context_features),
        'n_params': n_params,
        'test_corr_mse': test_metrics['corr_mse'],
        'test_r2': test_metrics['r2']
    }

In [ ]:
# Run experiments
results = []

# Experiment 1: Without freight (baseline)
results.append(run_experiment(CONTEXT_FEATURES_BASE, "Without freight (baseline)"))

# Experiment 2: With freight
results.append(run_experiment(CONTEXT_FEATURES_FREIGHT, "With freight"))

In [ ]:
# Summary
results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("SUMMARY: Freight Ablation Study")
print("="*60)
print(results_df.to_string(index=False))

# Compute difference
baseline = results_df[results_df['experiment'].str.contains('baseline')]['test_corr_mse'].values[0]
with_freight = results_df[results_df['experiment'].str.contains('With freight')]['test_corr_mse'].values[0]

diff_pct = (with_freight - baseline) / baseline * 100
print(f"\nDifference: {diff_pct:+.2f}%")
print(f"{'Freight HELPS' if diff_pct < 0 else 'Freight does NOT help'}")

In [ ]:
# Save results
results_df.to_csv('../tables/ablation_freight.csv', index=False)
print(f"\nSaved to tables/ablation_freight.csv")